<a href="https://colab.research.google.com/github/KatherineNietoP/Modulo_Python_maestria/blob/main/tarea_3_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Título del reto: Diseñar un flujo analítico reproducible

Se han obtenido bases de datos provenientes del Ministerio de Educación y del Instituto Nacional de Estadística y Censos (INEC). Para su integración, es necesario realizar una revisión y validación previa de las variables disponibles, con el fin de identificar aquellas que permitan establecer una clave común y garantizar una adecuada unión entre las diferentes fuentes de información.

In [ ]:
## Carga de paquetes

In [ ]:
import os
import sys
import hashlib
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np

## ** Creación de Rutas**

In [ ]:
base_tarea = Path("tarea_educacion")
raw_path = base_tarea / "data" / "raw"
proc_path = base_tarea / "data" / "processed"
raw_path.mkdir(parents=True, exist_ok=True)
proc_path.mkdir(parents=True, exist_ok=True)

# **4. Catálogo oficial de provincias**

In [ ]:


catalogo_provincias = pd.DataFrame({
    "cod_provincia": [
        "01", "02", "03", "04", "05", "06", "07", "08",
        "09", "10", "11", "12", "13", "14", "15", "16",
        "17", "18", "19", "20", "21", "22", "23", "24", "90"
    ],
    "provincia": [
        "Azuay",
        "Bolívar",
        "Cañar",
        "Carchi",
        "Cotopaxi",
        "Chimborazo",
        "El Oro",
        "Esmeraldas",
        "Guayas",
        "Imbabura",
        "Loja",
        "Los Ríos",
        "Manabí",
        "Morona Santiago",
        "Napo",
        "Pastaza",
        "Pichincha",
        "Tungurahua",
        "Zamora Chinchipe",
        "Galápagos",
        "Sucumbíos",
        "Orellana",
        "Santo Domingo",
        "Santa Elena",
        "Zona en estudio"
    ]
})

catalogo_provincias.to_csv(
    raw_path / "catalogo_provincias.csv",
    index=False,
    encoding="utf-8"
)

print(" Datos de prueba generados en `tarea_educacion/data/raw/`.")


 Datos de prueba generados en `tarea_educacion/data/raw/`.


# **2. LECTURA CONTROLADA**

Esta etapa permite cargar las diferentes fuentes de información de manera controlada, especificando el formato, separador, codificación y tipo de datos de las variables. Esto facilita una correcta interpretación de la información y prepara las bases para su posterior validación, transformación e integración.

In [ ]:
df_empleados = pd.read_excel(
    raw_path / "empleados.xlsx",
    dtype={"cod_prov": str}
)

df_socioeco = pd.read_csv(
    raw_path / "socioeconomico.csv",
    sep=";",
    dtype={"cod_prov": str},
    encoding="utf-8",
)



with open(raw_path / "educacion.json", "r", encoding="utf-8") as f:
    datos_educacion = json.load(f)

df_educacion = pd.json_normalize(datos_educacion)

print(df_empleados )
print(df_socioeco )
print(df_educacion )

   cod_prov         Provincia  total_empleados
0         1             AZUAY           302003
1         2           BOLIVAR            67836
2         3             CAÑAR            83165
3         4            CARCHI            65371
4         5          COTOPAXI           165099
5         6        CHIMBORAZO           189952
6         7            EL ORO           240214
7         8        ESMERALDAS           166796
8         9            GUAYAS          1385597
9        10          IMBABURA           161452
10       11              LOJA           165981
11       12          LOS RIOS           274997
12       13            MANABI           462691
13       14   MORONA SANTIAGO            53494
14       15              NAPO            39082
15       16           PASTAZA            31588
16       17         PICHINCHA          1182925
17       18        TUNGURAHUA           236305
18       19  ZAMORA CHINCHIPE            33779
19       20         GALAPAGOS            12446
20       21  

#3. CONTRATO DE CALIDAD

Esta etapa permite verificar que cada fuente cumpla con los requisitos mínimos de calidad antes de su integración. Se comprueba la existencia de las variables requeridas y que la variable utilizada como clave de unión no presente valores duplicados ni nulos, garantizando así una integración confiable de las bases de datos.

In [ ]:
print(df_empleados.columns)
print(df_socioeco.columns)
print(df_educacion.columns)

Index(['cod_prov', 'Provincia', 'total_empleados'], dtype='object')
Index(['cod_prov', 'Provincia', 'monto_prest', 'tipo_cuot'], dtype='object')
Index(['Cod_Prov', 'Provincia', 'Zonal', 'Total_Docentes',
       'Total_Estudiantes'],
      dtype='object')


In [ ]:
def validar_informacion(df: pd.DataFrame, nombre_fuente: str, columnas_requeridas: set, col_clave: str):
  errores = []

  faltantes = columnas_requeridas - set(df.columns)
  if faltantes:
    errores.append(f"Faltan columnas requeridas: {faltantes}")

  if col_clave in df.columns:
    if df[col_clave].duplicated().any():
        dups = df[col_clave][df[col_clave].duplicated()].tolist()
        errores.append(f"Claves duplicadas detectadas en {col_clave}: {dups}")
    if df[col_clave].isnull().any():
        errores.append(f"Existen valores nulos en la clave {col_clave}")

  if errores:
    raise ValueError(f" FALLÓ CONTRATO DE DATOS [{nombre_fuente}]: {'; '.join(errores)}")

  print(f" Contrato validado con éxito para [{nombre_fuente}] ({len(df)} registros)")
validar_informacion(df_socioeco, "socioeconomico CSV", {"cod_prov", "monto_prest", "tipo_cuot"}, "cod_prov")
validar_informacion(df_empleados, "empleados Excel", {"cod_prov", "total_empleados"}, "cod_prov")
validar_informacion(df_educacion, "educación JSON", {"Cod_Prov", "Zonal", "Total_Docentes",
       "Total_Estudiantes"}, "Cod_Prov")


 Contrato validado con éxito para [socioeconomico CSV] (22 registros)
 Contrato validado con éxito para [empleados Excel] (25 registros)
 Contrato validado con éxito para [educación JSON] (25 registros)


# 4. INTEGRACIÓN DE DATASET

En esta etapa se homogeneizan las variables clave de las diferentes fuentes para garantizar que puedan ser integradas correctamente. Primero, se unifica el nombre de la variable de identificación territorial como cod_provincia. Posteriormente, se estandariza su formato a dos dígitos, de acuerdo con la codificación de la División Político-Administrativa (DPA), mediante el uso de zfill(2). Esto permite que los códigos provinciales mantengan una estructura uniforme y sean utilizados como clave común para la integración de las bases.

Se va a renombrar las variables que se usara como ID codigo provincia.

In [ ]:
df_empleados = df_empleados.rename(columns={"cod_prov": "cod_provincia"})
df_socioeco = df_socioeco.rename(columns={"cod_prov": "cod_provincia"})
df_educacion = df_educacion.rename(columns={"Cod_Prov": "cod_provincia"})
df_educacion = df_educacion.rename(columns={"Cod_Prov": "cod_provincia"})


Se estandariza la variable cod_provincia a dos digitos como se establece en la DPA

In [ ]:
df_empleados["cod_provincia"] = df_empleados["cod_provincia"].astype(str).str.zfill(2)
df_socioeco["cod_provincia"] = df_socioeco["cod_provincia"].astype(str).str.zfill(2)
df_educacion["cod_provincia"] = df_educacion["cod_provincia"].astype(str).str.zfill(2)

Se realiza la integración progresiva de las diferentes fuentes utilizando cod_provincia como clave común de unión. Se emplea un left join tomando como base el catálogo de provincias, con el propósito de conservar todas las provincias registradas en el catálogo e incorporar la información disponible de las bases de empleados, socioeconómica y educación.

Asimismo, mediante indicator se generan variables de control (_merge_empleo, _merge_prestamo y _merge_educacion) que permiten identificar posteriormente si los registros de cada fuente fueron correctamente vinculados al consolidado.

In [ ]:
merge_1 = catalogo_provincias.merge(
    df_empleados,
    on="cod_provincia",
    how="left",
    indicator="_merge_empleo"
)

merge_2 = merge_1.merge(
    df_socioeco,
    on="cod_provincia",
    how="left",
    indicator="_merge_prestamo"
)

consolidado = merge_2.merge(
    df_educacion,
    on="cod_provincia",
    how="left",
    indicator="_merge_educacion"
)


Esta etapa permite verificar el resultado de las uniones realizadas entre las bases de datos. Mediante las variables de control generadas con indicator, se identifica cuántos registros del catálogo de provincias encontraron correspondencia en cada fuente y cuáles no presentaron coincidencias. Esto permite detectar posibles inconsistencias o ausencia de información antes de continuar con el análisis.

In [ ]:
print("AUDITORÍA DE COINCIDENCIAS")
print("\n")
print("Presencia en empleo:")
print(consolidado["_merge_empleo"].value_counts())
print("\n")
print("Presencia en socioeconomico:")
print(consolidado["_merge_prestamo"].value_counts())
print("Presencia en educacion:")
print(consolidado["_merge_educacion"].value_counts())


AUDITORÍA DE COINCIDENCIAS


Presencia en empleo:
_merge_empleo
both          25
left_only      0
right_only     0
Name: count, dtype: int64


Presencia en socioeconomico:
_merge_prestamo
both          22
left_only      3
right_only     0
Name: count, dtype: int64
Presencia en educacion:
_merge_educacion
both          25
left_only      0
right_only     0
Name: count, dtype: int64


Una vez finalizada la auditoría de coincidencias, se eliminan las columnas auxiliares de control generadas durante los merge (_merge_). Estas variables fueron utilizadas únicamente para verificar la correcta integración de las fuentes, por lo que no son necesarias en la base final.



In [ ]:
columnas_indicadoras = [c for c in consolidado.columns if c.startswith("_merge_")]
consolidado_limpio = consolidado.drop(columns=columnas_indicadoras)


In [ ]:
display(consolidado_limpio)

,cod_provincia,provincia,Provincia_x,total_empleados,Provincia_y,monto_prest,tipo_cuot,Provincia,Zonal,Total_Docentes,Total_Estudiantes
0,01,Azuay,AZUAY,302003,Azuay,"2291410,99",Cuota fija,AZUAY,Zona 6,11130,189698
1,02,Bolívar,BOLIVAR,67836,Bolivar,654425,Cuota fija,BOLIVAR,Zona 5,2896,46893
2,03,Cañar,CAÑAR,83165,Cañar,"843836,71",Cuota fija,CAÑAR,Zona 6,3222,57407
3,04,Carchi,CARCHI,65371,Carchi,743820,Cuota fija,CARCHI,Zona 1,2384,38444
4,05,Cotopaxi,COTOPAXI,165099,Cotopaxi,1304926,Cuota variable,COTOPAXI,Zona 3,6259,111834
5,06,Chimborazo,CHIMBORAZO,189952,Chimborazo,930375,Cuota fija,CHIMBORAZO,Zona 3,7149,108868
6,07,El Oro,EL ORO,240214,El Oro,1664224,Cuota fija,EL ORO,Zona 7,8876,166937
7,08,Esmeraldas,ESMERALDAS,166796,Esmeraldas,401700,Cuota fija,ESMERALDAS,Zona 1,8228,157657
8,09,Guayas,GUAYAS,1385597,Guayas,"1253599,6",Cuota fija,GUAYAS,Zona 5,46185,1038829
9,10,Imbabura,IMBABURA,161452,Imbabura,"755558,99",Cuota fija,IMBABURA,Zona 1,6499,115171


Solo mantenemos la variable de provincia de la base de catalogo

In [ ]:
consolidado_limpio = consolidado_limpio.drop(columns=["Provincia_x", "Provincia_y","Provincia"], errors="ignore")


In [ ]:
display(consolidado_limpio)

,cod_provincia,provincia,total_empleados,monto_prest,tipo_cuot,Zonal,Total_Docentes,Total_Estudiantes
0,01,Azuay,302003,"2291410,99",Cuota fija,Zona 6,11130,189698
1,02,Bolívar,67836,654425,Cuota fija,Zona 5,2896,46893
2,03,Cañar,83165,"843836,71",Cuota fija,Zona 6,3222,57407
3,04,Carchi,65371,743820,Cuota fija,Zona 1,2384,38444
4,05,Cotopaxi,165099,1304926,Cuota variable,Zona 3,6259,111834
5,06,Chimborazo,189952,930375,Cuota fija,Zona 3,7149,108868
6,07,El Oro,240214,1664224,Cuota fija,Zona 7,8876,166937
7,08,Esmeraldas,166796,401700,Cuota fija,Zona 1,8228,157657
8,09,Guayas,1385597,"1253599,6",Cuota fija,Zona 5,46185,1038829
9,10,Imbabura,161452,"755558,99",Cuota fija,Zona 1,6499,115171


Verificamos en que variables existe NA

In [ ]:
consolidado_limpio.isnull().sum()

,0
cod_provincia,0
provincia,0
total_empleados,0
monto_prest,3
tipo_cuot,3
Zonal,0
Total_Docentes,0
Total_Estudiantes,0


En esta etapa se identifican y tratan los valores faltantes de las variables seleccionadas. Para monto_pres los valores NaN se reemplazan por 0, y las de tipo categorica tipo_cuot con "sin información", con el propósito de evitar valores ausentes en el consolidado final

In [ ]:
consolidado_limpio["monto_prest"] = consolidado_limpio["monto_prest"].fillna(0)

consolidado_limpio["tipo_cuot"] = consolidado_limpio["tipo_cuot"].fillna("Sin información")
display(consolidado_limpio)

,cod_provincia,provincia,total_empleados,monto_prest,tipo_cuot,Zonal,Total_Docentes,Total_Estudiantes
0,01,Azuay,302003,"2291410,99",Cuota fija,Zona 6,11130,189698
1,02,Bolívar,67836,654425,Cuota fija,Zona 5,2896,46893
2,03,Cañar,83165,"843836,71",Cuota fija,Zona 6,3222,57407
3,04,Carchi,65371,743820,Cuota fija,Zona 1,2384,38444
4,05,Cotopaxi,165099,1304926,Cuota variable,Zona 3,6259,111834
5,06,Chimborazo,189952,930375,Cuota fija,Zona 3,7149,108868
6,07,El Oro,240214,1664224,Cuota fija,Zona 7,8876,166937
7,08,Esmeraldas,166796,401700,Cuota fija,Zona 1,8228,157657
8,09,Guayas,1385597,"1253599,6",Cuota fija,Zona 5,46185,1038829
9,10,Imbabura,161452,"755558,99",Cuota fija,Zona 1,6499,115171


# **6. Variable Derivada**

Se calcula la tasa de docentes por cada 100 estudiantes, relacionando el número total de docentes con el número total de estudiantes. Se utiliza np.where para evitar divisiones por cero en los casos en que no existan estudiantes registrados.

In [ ]:
consolidado_limpio["tasa_docente_estu"] = np.where(
    consolidado_limpio["Total_Estudiantes"] > 0,
    (consolidado_limpio["Total_Docentes"] / consolidado_limpio["Total_Estudiantes"]) * 100,
    0.0
)
print("Tasa de docentes por estudiante:")
print(consolidado_limpio["tasa_docente_estu"])

Tasa de docentes por estudiante:
0      5.867221
1      6.175762
2      5.612556
3      6.201228
4      5.596688
5      6.566668
6      5.316976
7      5.218925
8      4.445871
9      5.642914
10     7.043683
11     4.366500
12     5.260129
13     5.363848
14     6.748989
15     5.932252
16     5.640759
17     5.317865
18     6.755405
19     6.657609
20     5.625674
21     5.432099
22     4.680711
23     4.553324
24    10.893855
Name: tasa_docente_estu, dtype: float64


# **7. Auditoria**

En esta etapa se guardan los resultados del procesamiento en formatos CSV y Excel, organizados en las carpetas correspondientes de datos procesados y productos finales. Posteriormente, se calcula el hash SHA-256 de cada archivo generado, lo que permite verificar su integridad y detectar posibles modificaciones posteriores a su generación.

Además, la función calcular_sha256() garantiza que cada archivo tenga una huella digital única que puede utilizarse para fines de trazabilidad y control de los resultados.

In [ ]:
from pathlib import Path

processed_path = Path("/content/drive/MyDrive/taller_educacion/data/processed")
outputs_path = Path("/content/drive/MyDrive/taller_educacion/data/outputs")

processed_path.mkdir(parents=True, exist_ok=True)
outputs_path.mkdir(parents=True, exist_ok=True)

ruta_salida_csv = processed_path / "consolidado_provincial_informacion_202601.csv"
ruta_salida_excel = outputs_path / "tablero_indicadores_informacion_202601.xlsx"

consolidado_limpio.to_csv(
    ruta_salida_csv,
    index=False,
    encoding="utf-8"
)

consolidado_limpio.to_excel(
    ruta_salida_excel,
    index=False
)

def calcular_sha256(ruta_archivo: Path) -> str:
  hasher = hashlib.sha256()
  with open(ruta_archivo, "rb") as f:
    hasher.update(f.read())
  return hasher.hexdigest()

hash_csv = calcular_sha256(ruta_salida_csv)
hash_excel = calcular_sha256(ruta_salida_excel)




El manifiesto permite mantener una trazabilidad básica del procesamiento y de los archivos generados, facilitando su identificación y posterior verificación.

In [ ]:
import json

manifiesto = {
    "archivo_procesado": ruta_salida_csv.name,
    "archivo_excel": ruta_salida_excel.name,
    "filas_totales": len(consolidado_limpio),
    "columnas_totales": len(consolidado_limpio.columns),
    "sha256_csv": hash_csv,
    "sha256_excel": hash_excel
}

ruta_manifiesto = outputs_path / "manifiesto_ejecucion.json"

with open(ruta_manifiesto, "w", encoding="utf-8") as f:
    json.dump(
        manifiesto,
        f,
        ensure_ascii=False,
        indent=4
    )

print(f"Manifiesto guardado en: {ruta_manifiesto}")

Manifiesto guardado en: /content/drive/MyDrive/taller_educacion/data/outputs/manifiesto_ejecucion.json


In [ ]:
print(" Proceso finalizado con éxito.")
print(f" - CSV SHA256: {hash_csv}")
print(f" - EXCEL SHA256: {hash_excel}")
print(f" - Manifiesto guardado en: {ruta_manifiesto}")

 Proceso finalizado con éxito.
 - CSV SHA256: 78e2ab3365cbafc1e1c18b114ee7efb9cf29197476eb58ec15620e872e5ab7ee
 - EXCEL SHA256: 8ef776353dc944d7158d336a44ead76b657b73a02d933da1dfbbbaddb81ee566
 - Manifiesto guardado en: /content/drive/MyDrive/taller_educacion/data/outputs/manifiesto_ejecucion.json


Esta etapa confirma la finalización exitosa del flujo de procesamiento y presenta los principales elementos de control generados. Se muestran los códigos SHA-256 correspondientes a los archivos CSV y Excel, así como la ubicación del manifiesto de ejecución, permitiendo verificar y mantener la trazabilidad de los productos finales.